In [8]:
import os
import time
import random
import traceback
import pandas as pd
from vnstock import Quote

In [ ]:
# --------------------------------------------------
# Config
# --------------------------------------------------
home = "./data"
os.makedirs(home, exist_ok=True)

symbols = [
    # Banking
    "VCB","BID","CTG","TCB","MBB","ACB","VPB","STB","HDB",
    "VIB","TPB","LPB","SHB",

    # Real estate
    "VHM","VIC","VRE","VPL",

    # Industry
    "HPG","HSG", "DGC","GVR",

    # Consumer
    "MWG","MSN","VNM","SAB","PNJ",

    # Energy
    "POW","GAS","PLX",

    # Others
    "FPT","SSI","VJC","BVH"
]

start = "2018-01-01"
end   = "2026-03-15"

# KBS is usually easier on Colab/Kaggle; 
# VCI can be more complete on local machines
source = "VCI"

In [6]:
def normalize_history(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """
    Standardize vnstock history output to:
    date, symbol, open, high, low, close, volume
    """
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["date", "symbol", "open", "high", "low", "close", "volume"])

    out = df.copy()

    out = out.rename(columns={"time": "date"})

    required_price_cols = ["open", "high", "low", "close", "volume"]
    missing = [c for c in required_price_cols if c not in out.columns]
    if missing:
        raise ValueError(f"{symbol}: missing expected columns {missing}. Got: {list(out.columns)}")

    out["date"] = pd.to_datetime(out["date"]).dt.tz_localize(None)
    out["symbol"] = symbol

    out = out[["date", "symbol", "open", "high", "low", "close", "volume"]].copy()
    out = out.sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)
    return out


def download_one_symbol(symbol: str, start: str, end: str, source: str) -> pd.DataFrame:
    """
    Download one symbol from vnstock.
    """
    q = Quote(symbol=symbol, source=source)
    df = q.history(start=start, end=end, interval="1D")
    return normalize_history(df, symbol)


def save_symbol_csv(df: pd.DataFrame, folder: str, symbol: str):
    path = os.path.join(folder, f"{symbol}.csv")
    df.to_csv(path, index=False)
    print(f"Saved {symbol}: {len(df):,} rows -> {path}")


In [11]:
all_frames = []
failed = []

for i, symbol in enumerate(symbols, 1):
    try:
        print(f"[{i}/{len(symbols)}] Downloading {symbol} from {source} ...")
        df_sym = download_one_symbol(symbol, start=start, end=end, source=source)

        if df_sym.empty:
            print(f"  -> Empty data for {symbol}")
            failed.append((symbol, "empty dataframe"))
            continue

        save_symbol_csv(df_sym, home, symbol)
        all_frames.append(df_sym)

        # be polite to source
        time.sleep(0.8 + random.random() * 0.8)

    except Exception as e:
        print(f"  -> Failed {symbol}: {e}")
        failed.append((symbol, str(e)))
        # optional debug:
        # traceback.print_exc()
        time.sleep(1.5)

[1/14] Downloading DGC from VCI ...
Saved DGC: 2,142 rows -> ./data\DGC.csv
[2/14] Downloading GVR from VCI ...
Saved GVR: 1,987 rows -> ./data\GVR.csv
[3/14] Downloading MWG from VCI ...
Saved MWG: 2,142 rows -> ./data\MWG.csv
[4/14] Downloading MSN from VCI ...
Saved MSN: 2,142 rows -> ./data\MSN.csv
[5/14] Downloading VNM from VCI ...
Saved VNM: 2,142 rows -> ./data\VNM.csv
[6/14] Downloading SAB from VCI ...
Saved SAB: 2,142 rows -> ./data\SAB.csv
[7/14] Downloading PNJ from VCI ...
Saved PNJ: 2,142 rows -> ./data\PNJ.csv
[8/14] Downloading POW from VCI ...
Saved POW: 1,995 rows -> ./data\POW.csv
[9/14] Downloading GAS from VCI ...
Saved GAS: 2,142 rows -> ./data\GAS.csv
[10/14] Downloading PLX from VCI ...
Saved PLX: 2,142 rows -> ./data\PLX.csv
[11/14] Downloading FPT from VCI ...
Saved FPT: 2,142 rows -> ./data\FPT.csv
[12/14] Downloading SSI from VCI ...
Saved SSI: 2,142 rows -> ./data\SSI.csv
[13/14] Downloading VJC from VCI ...
Saved VJC: 2,142 rows -> ./data\VJC.csv
[14/14] 

In [13]:
# --------------------------------------------------
# Save failures
# --------------------------------------------------
if failed:
    failed_df = pd.DataFrame(failed, columns=["symbol", "error"])
    print(failed_df)
else:
    print("\nAll symbols downloaded successfully.")


All symbols downloaded successfully.


In [15]:
# --------------------------------------------------
# Save combined panel
# --------------------------------------------------
if all_frames:
    panel = pd.concat(all_frames, ignore_index=True)
    panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)

    print(panel.head())

    coverage = (
        panel.groupby("symbol")["date"]
        .agg(["min", "max", "count"])
        .sort_values("min")
        .reset_index()
    )

    print("\nCoverage summary:")
    print(coverage)
    print(f"\nCoverage saved: {coverage_path}")
else:
    print("\nNo data downloaded.")


        date symbol   open   high    low  close  volume
0 2017-08-15    BVH  47.02  47.27  46.94  46.94  183940
1 2017-08-16    BVH  46.77  47.02  46.52  47.02  237440
2 2017-08-17    BVH  47.10  47.10  46.52  46.69  197450
3 2017-08-18    BVH  46.69  46.77  46.44  46.61  195000
4 2017-08-21    BVH  46.61  46.61  45.44  45.69  368370

Coverage summary:
   symbol        min        max  count
0     DGC 2017-08-07 2026-03-13   2142
1     BVH 2017-08-15 2026-03-13   2142
2     FPT 2017-08-15 2026-03-13   2142
3     GAS 2017-08-15 2026-03-13   2142
4     MSN 2017-08-15 2026-03-13   2142
5     MWG 2017-08-15 2026-03-13   2142
6     PLX 2017-08-15 2026-03-13   2142
7     PNJ 2017-08-15 2026-03-13   2142
8     SAB 2017-08-15 2026-03-13   2142
9     SSI 2017-08-15 2026-03-13   2142
10    VJC 2017-08-15 2026-03-13   2142
11    VNM 2017-08-15 2026-03-13   2142
12    POW 2018-03-06 2026-03-13   1995
13    GVR 2018-03-21 2026-03-13   1987

Coverage saved: ./data\coverage_summary.csv
